# shared embedding artifacts

In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!python --version
!python -m pip --version
!python -c "import platform; print(platform.platform())"

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)


Python 3.12.13
pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Linux-6.6.122+-x86_64-with-glibc2.35
torch: 2.10.0+cpu
cuda available: False
cuda version: None


In [3]:
from pathlib import Path

DRIVE_WORK_DIR = Path("/content/drive/MyDrive/variable_misuse_graph_build")

# Đổi 300_000 nếu bạn tạo subset kích thước khác.
SUBSET_TOTAL_SAMPLES = 300_000

PROCESSED_SUBSET_ROOT = (
    DRIVE_WORK_DIR
    / "dataset"
    / "processed"
    / f"great_colab_balanced_{SUBSET_TOTAL_SAMPLES}"
)

TRAIN_JSONL = PROCESSED_SUBSET_ROOT / "train.jsonl"

EMBEDDING_OUTPUT_ROOT = (
    DRIVE_WORK_DIR
    / "artifacts"
    / "embedding"
    / f"shared_token_embedding_{SUBSET_TOTAL_SAMPLES}"
)

EMBEDDING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("TRAIN_JSONL:", TRAIN_JSONL, TRAIN_JSONL.exists())
print("EMBEDDING_OUTPUT_ROOT:", EMBEDDING_OUTPUT_ROOT)

assert TRAIN_JSONL.exists(), f"Không tìm thấy train subset: {TRAIN_JSONL}"


TRAIN_JSONL: /content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000/train.jsonl True
EMBEDDING_OUTPUT_ROOT: /content/drive/MyDrive/variable_misuse_graph_build/artifacts/embedding/shared_token_embedding_300000


In [4]:
EMBEDDING_CONFIG = {
    "source_train_jsonl": TRAIN_JSONL.as_posix(),
    "subset_total_samples": SUBSET_TOTAL_SAMPLES,
    "vocab_type": "token_level",
    "max_vocab_size": 50_000,
    "min_frequency": 2,
    "embedding_dim": 128,
    "seed": 42,
    "special_tokens": {
        "pad_token": "<PAD>",
        "unk_token": "<UNK>",
        "no_bug_token": "<MASK_NO_BUG>",
    },
}

PAD_TOKEN = EMBEDDING_CONFIG["special_tokens"]["pad_token"]
UNK_TOKEN = EMBEDDING_CONFIG["special_tokens"]["unk_token"]
NO_BUG_TOKEN = EMBEDDING_CONFIG["special_tokens"]["no_bug_token"]

print(EMBEDDING_CONFIG)


{'source_train_jsonl': '/content/drive/MyDrive/variable_misuse_graph_build/dataset/processed/great_colab_balanced_300000/train.jsonl', 'subset_total_samples': 300000, 'vocab_type': 'token_level', 'max_vocab_size': 50000, 'min_frequency': 2, 'embedding_dim': 128, 'seed': 42, 'special_tokens': {'pad_token': '<PAD>', 'unk_token': '<UNK>', 'no_bug_token': '<MASK_NO_BUG>'}}


In [5]:
import json
from collections import Counter

token_counter = Counter()
sample_count = 0
token_count = 0

with TRAIN_JSONL.open("r", encoding="utf-8") as file:
    for line in file:
        item = json.loads(line)
        tokens = item["tokens"]
        token_counter.update(tokens)
        sample_count += 1
        token_count += len(tokens)

print("samples:", sample_count)
print("tokens:", token_count)
print("unique tokens:", len(token_counter))
print("top 30 tokens:")
for token, count in token_counter.most_common(30):
    print(repr(token), count)


samples: 182678
tokens: 18558029
unique tokens: 590290
top 30 tokens:
'#NEWLINE#' 1863923
')' 1726609
'(' 1533589
'.' 1451629
',' 1137610
'=' 829493
'self' 661011
':' 623059
'#INDENT#' 561887
'#UNINDENT#' 327053
'[' 295342
']' 295342
'if' 170157
'return' 122604
'None' 94273
'in' 85108
'0' 78433
'1' 77215
'for' 65391
'{' 57032
'}' 57032
'else' 54712
'==' 52908
'name' 46926
'+' 43946
'assertEqual' 42780
'True' 38897
'%' 34897
'data' 33731
'*' 33708


In [6]:
MAX_VOCAB_SIZE = EMBEDDING_CONFIG["max_vocab_size"]
MIN_FREQUENCY = EMBEDDING_CONFIG["min_frequency"]

special_tokens = [PAD_TOKEN, UNK_TOKEN, NO_BUG_TOKEN]

token_to_id = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
    NO_BUG_TOKEN: 2,
}

for token, count in token_counter.most_common():
    if count < MIN_FREQUENCY:
        continue
    if token in token_to_id:
        continue
    if len(token_to_id) >= MAX_VOCAB_SIZE:
        break
    token_to_id[token] = len(token_to_id)

id_to_token = {str(index): token for token, index in token_to_id.items()}

covered_count = sum(
    count for token, count in token_counter.items()
    if token in token_to_id
)
oov_count = token_count - covered_count

vocab_summary = {
    "sample_count": sample_count,
    "token_count": token_count,
    "unique_tokens_before_cutoff": len(token_counter),
    "vocab_size": len(token_to_id),
    "max_vocab_size": MAX_VOCAB_SIZE,
    "min_frequency": MIN_FREQUENCY,
    "covered_token_count": covered_count,
    "oov_token_count": oov_count,
    "coverage": covered_count / token_count if token_count else 0.0,
    "top_tokens": token_counter.most_common(100),
}

print("vocab_size:", len(token_to_id))
print("coverage:", vocab_summary["coverage"])
print("oov_count:", oov_count)
print("special ids:", {token: token_to_id[token] for token in special_tokens})


vocab_size: 50000
coverage: 0.9364737494482847
oov_count: 1178922
special ids: {'<PAD>': 0, '<UNK>': 1, '<MASK_NO_BUG>': 2}


In [7]:
import random
import numpy as np
import torch

seed = EMBEDDING_CONFIG["seed"]
embedding_dim = EMBEDDING_CONFIG["embedding_dim"]
vocab_size = len(token_to_id)

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

embedding_init = torch.empty(vocab_size, embedding_dim)
torch.nn.init.normal_(embedding_init, mean=0.0, std=0.02)

# PAD embedding nên là vector 0.
embedding_init[token_to_id[PAD_TOKEN]].zero_()

print("embedding_init shape:", tuple(embedding_init.shape))
print("PAD norm:", embedding_init[token_to_id[PAD_TOKEN]].norm().item())
print("UNK norm:", embedding_init[token_to_id[UNK_TOKEN]].norm().item())


embedding_init shape: (50000, 128)
PAD norm: 0.0
UNK norm: 0.2297874242067337


In [8]:
import json

token_to_id_path = EMBEDDING_OUTPUT_ROOT / "token_to_id.json"
id_to_token_path = EMBEDDING_OUTPUT_ROOT / "id_to_token.json"
config_path = EMBEDDING_OUTPUT_ROOT / "embedding_config.json"
summary_path = EMBEDDING_OUTPUT_ROOT / "vocabulary_summary.json"
embedding_path = EMBEDDING_OUTPUT_ROOT / "embedding_init.pt"

token_to_id_path.write_text(
    json.dumps(token_to_id, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

id_to_token_path.write_text(
    json.dumps(id_to_token, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

config_to_save = dict(EMBEDDING_CONFIG)
config_to_save["vocab_size"] = vocab_size
config_to_save["pad_idx"] = token_to_id[PAD_TOKEN]
config_to_save["unk_idx"] = token_to_id[UNK_TOKEN]
config_to_save["no_bug_idx"] = token_to_id[NO_BUG_TOKEN]

config_path.write_text(
    json.dumps(config_to_save, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

summary_path.write_text(
    json.dumps(vocab_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

torch.save(
    {
        "embedding_weight": embedding_init,
        "config": config_to_save,
    },
    embedding_path,
)

print("Saved artifacts to:", EMBEDDING_OUTPUT_ROOT)
!find "{EMBEDDING_OUTPUT_ROOT}" -maxdepth 1 -type f -printf "%f %s bytes\n"


Saved artifacts to: /content/drive/MyDrive/variable_misuse_graph_build/artifacts/embedding/shared_token_embedding_300000
token_to_id.json 1135411 bytes
id_to_token.json 1235411 bytes
embedding_config.json 479 bytes
vocabulary_summary.json 4138 bytes
embedding_init.pt 25602138 bytes


In [10]:
loaded_token_to_id = json.loads(token_to_id_path.read_text(encoding="utf-8"))
loaded_embedding = torch.load(embedding_path, map_location="cpu")

assert loaded_token_to_id[PAD_TOKEN] == 0
assert loaded_token_to_id[UNK_TOKEN] == 1
assert loaded_token_to_id[NO_BUG_TOKEN] == 2
assert tuple(loaded_embedding["embedding_weight"].shape) == tuple(embedding_init.shape)

print("Verify OK")


Verify OK


In [11]:
import zipfile

zip_path = EMBEDDING_OUTPUT_ROOT.parent / f"{EMBEDDING_OUTPUT_ROOT.name}.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in EMBEDDING_OUTPUT_ROOT.glob("*"):
        archive.write(path, arcname=path.name)

print("Created:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))


Created: /content/drive/MyDrive/variable_misuse_graph_build/artifacts/embedding/shared_token_embedding_300000.zip
Size MB: 23.41
